# BirdCLEF 2026 Inference (v3)
Minimal, hard-coded Kaggle-ready inference: loads model/labels from the specified dataset, scores test soundscapes in 5s windows, and writes /kaggle/working/submission.csv.

In [ ]:
# Minimal, hard-coded Kaggle-ready inference: loads model/labels from the specified dataset, scores test soundscapes in 5s windows, and writes /kaggle/working/submission.csv."""
import json, time
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import torch
import torch.nn as nn

MODEL_FILE = Path('/kaggle/input/models/robertjaret/birdclef-2026-cnnmodel-231706/pytorch/default/1')
SAMPLE_SUB = Path('/kaggle/input/competitions/birdclef-2026/sample_submission.csv')
TEST_DIR = Path('/kaggle/input/competitions/birdclef-2026/test_soundscapes')
OUT = Path('/kaggle/working/submission.csv')
LABELS_PATH = Path('/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
JSON_LABELS_PATH = Path('/kaggle/input/datasets/robertjaret/birdclef-2026-model-labels-v2-json')

# MODEL_FILE = Path('checkpoints/best_model_20260603_231706.pt')
# SAMPLE_SUB = Path('data/sample_submission.csv')
# TEST_DIR = Path('data/test_soundscapes')
# OUT = Path('results/submission.csv')
# LABELS_PATH = Path('data/taxonomy.csv')
# JSON_LABELS_PATH = Path('artifacts/model_labels_v2.json')

# Audio / spectrogram params
SAMPLE_RATE = 32000
WINDOW_SECONDS = 5
WINDOW_SAMPLES = SAMPLE_RATE * WINDOW_SECONDS
N_MELS = 128
N_FFT = 1024
HOP = 320
FMIN = 50.0
FMAX = 14000.0
BATCH = 64

# small, simple CNN
class BirdClefCNNModel(nn.Module):
    def __init__(self, n_mels=None, time_steps=None, num_classes=None, dropout=0.3):
        super(BirdClefCNNModel, self).__init__()
        self.model_name = "BirdClefCNNModel"
        if n_mels is None:
            raise ValueError("n_mels must be provided either in h dict or as parameter")
        if time_steps is None:
            raise ValueError("time_steps must be provided as parameter")
        self.conv1 = nn.Conv2d(1, 32, kernel_size=(3, 3), padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=(3, 3), padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=(3, 3), padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        # Validate input size for pooling
        min_time_steps = 2 ** 3  # 8 for 3 poolings
        if time_steps < min_time_steps:
            raise ValueError(f"time_steps must be at least {min_time_steps} for this model, got {time_steps}")

        # Global average pooling + simple classifier
        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        # x: (batch, time_steps, n_mels)
        # Reshape to (batch, 1, n_mels, time_steps)
        x = x.permute(0, 2, 1).unsqueeze(1)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.pool(x)
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.pool(x)
        # Global average pooling: (batch, 128, h, w) -> (batch, 128, 1, 1)
        x = self.global_avg_pool(x)
        x = x.view(x.size(0), -1)  # (batch, 128)
        x = self.fc(x)
        return x


def compute_log_mel_spectrogram(wave, sr):
    # uses librosa for simplicity; returns time x n_mels (float32)
    S = librosa.feature.melspectrogram(y=wave, sr=sr, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS, fmin=FMIN, fmax=FMAX, power=2.0)
    log_S = np.log(S + 1e-6)
    return log_S.T.astype('float32')


def load_audio(path):
    x, sr = sf.read(str(path), dtype='float32')
    if sr != SAMPLE_RATE:
        x = librosa.resample(x, orig_sr=sr, target_sr=SAMPLE_RATE)
    if x.ndim > 1:
        x = x.mean(axis=1)
    return x.astype('float32')

try:

    if not MODEL_FILE.exists():
        raise FileNotFoundError(f'MODEL_FILE not found: {MODEL_FILE}')
    # If the MODEL_FILE path is a directory (Kaggle Models often expose a directory),
    # try to find a sensible model file inside it and replace MODEL_FILE with that path.
    if MODEL_FILE.is_dir():
        candidates = [p for p in MODEL_FILE.iterdir() if p.is_file()]
        if not candidates:
            candidates = [p for p in MODEL_FILE.rglob('*') if p.is_file()]
        if not candidates:
            raise FileNotFoundError(f'No model files found in directory: {MODEL_FILE}')
        exts = ['.pt', '.pth', '.ckpt', '.bin']
        candidates.sort(key=lambda p: (0 if p.suffix in exts else 1, p.name))
        MODEL_FILE = candidates[0]
    if not SAMPLE_SUB.exists():
        raise FileNotFoundError(f'sample_submission.csv not found: {SAMPLE_SUB}')

    # load checkpoint and robustly extract state_dict
    ck = torch.load(str(MODEL_FILE), map_location='cpu')

    def _looks_like_state_dict(d):
        if not isinstance(d, dict):
            return False
        # look for keys containing '.' and values that are tensors/ndarrays
        keys = list(d.keys())[:10]
        if not keys:
            return False
        if not all(isinstance(k, str) for k in keys):
            return False
        if not any('.' in k for k in keys):
            return False
        vals = list(d.values())[:10]
        return any(hasattr(v, 'shape') for v in vals)

    state = None
    if isinstance(ck, dict):
        if _looks_like_state_dict(ck):
            state = ck
        else:
            # search nested dicts for a plausible state_dict
            for v in ck.values():
                if _looks_like_state_dict(v):
                    state = v
                    break
    if state is None:
        # last resort: treat ck as state
        state = ck

    # normalize common prefix like 'module.' or 'model.' if present
    prefix = None
    for k in list(state.keys())[:20]:
        if k.startswith('module.'):
            prefix = 'module.'
            break
        if k.startswith('model.'):
            prefix = 'model.'
            break
    if prefix is not None:
        state = {k[len(prefix):] if k.startswith(prefix) else k: v for k, v in state.items()}

    # find fc weight key by suffix
    def _find_by_suffix(sd, suffix):
        for k in sd.keys():
            if k.endswith(suffix):
                return k
        return None

    wkey = _find_by_suffix(state, 'fc.weight')
    bkey = _find_by_suffix(state, 'fc.bias')

    ck_num_classes = None
    if wkey is not None:
        try:
            ck_num_classes = int(state[wkey].shape[0])
        except Exception:
            ck_num_classes = None

    # Try to get labels from checkpoint metadata if available
    labels = None
    if isinstance(ck, dict):
        for k in ('labels', 'model_labels', 'class_labels', 'classes', 'label_list'):
            if k in ck and isinstance(ck[k], (list, tuple)):
                labels = [str(x) for x in ck[k]]
                break
        if labels is None and 'h' in ck and isinstance(ck['h'], dict):
            for k in ('labels', 'model_labels', 'classes', 'label_list'):
                if k in ck['h'] and isinstance(ck['h'][k], (list, tuple)):
                    labels = [str(x) for x in ck['h'][k]]
                    break

    # Fallback to taxonomy.csv if that JSON is missing; avoid exhaustive directory searches.
    if labels is None:
        ml_file = JSON_LABELS_PATH
        if ml_file.exists():
            try:
                labels = json.loads(ml_file.read_text())
                labels = [str(x) for x in labels]
            except Exception:
                labels = None

    # Last resort: taxonomy.csv from competition input
    if labels is None and LABELS_PATH.exists():
        df_labels = pd.read_csv(str(LABELS_PATH))
        labels = df_labels.iloc[:, 0].astype(str).tolist()

    if labels is None:
        raise FileNotFoundError('Could not determine model labels from checkpoint, JSON, or taxonomy.csv')

    # Prefer num_classes from checkpoint fc if available; otherwise taxonomy
    if ck_num_classes is not None:
        num_classes = ck_num_classes
        print(f'Using num_classes from checkpoint fc: {num_classes}')
    else:
        num_classes = len(labels)
        print(f'Using num_classes from labels/taxonomy: {num_classes}')

    # If the taxonomy labels cover more classes than the checkpoint's final layer,
    # assume the model was trained on the first ck_num_classes entries (common),
    # so truncate taxonomy to match ck_num_classes. If taxonomy is shorter, pad with placeholders.
    if ck_num_classes is not None and len(labels) != ck_num_classes:
        if len(labels) > ck_num_classes:
            print(f'Warning: taxonomy contains {len(labels)} labels but checkpoint has {ck_num_classes} outputs; truncating labels to first {ck_num_classes}.')
            labels = labels[:ck_num_classes]
        else:
            print(f'Warning: taxonomy contains {len(labels)} labels but checkpoint has {ck_num_classes} outputs; padding labels with placeholders to match.')
            labels = labels + [f'__missing_label_{i}' for i in range(len(labels), ck_num_classes)]

    sample = pd.read_csv(str(SAMPLE_SUB)).head(0)
    submission_cols = [c for c in sample.columns if c != 'row_id']

    # Build mapping as intersection (no clamping) and record unmapped columns.
    mapping = {c: labels.index(c) for c in submission_cols if c in labels}
    unmapped = [c for c in submission_cols if c not in mapping]
    if unmapped:
        print(f'Warning: {len(unmapped)} submission columns not found in labels: {unmapped[:5]}{"..." if len(unmapped)>5 else ""}')
    print(f'num_classes={num_classes}, len(labels)={len(labels)}, submission_cols={len(submission_cols)}, mapped={len(mapping)}, unmapped={len(unmapped)}')

    model = BirdClefCNNModel(n_mels=N_MELS, time_steps=997, num_classes=num_classes)
    model.load_state_dict(state, strict=False)
    model.eval()

    if not TEST_DIR.is_dir():
        # Graceful behavior for local testing: write header-only submission
        print(f'No test soundscapes at {TEST_DIR}; writing header-only submission to {OUT}')
        df_empty = pd.DataFrame(columns=['row_id'] + submission_cols)
        OUT.parent.mkdir(parents=True, exist_ok=True)
        df_empty.to_csv(str(OUT), index=False)
        print(f'Wrote empty submission {OUT} (0 rows)')
    else:
        test_files = sorted(TEST_DIR.glob('*.ogg'))
        rows = []
        t0 = time.time()
        for fp in test_files:
            wav = load_audio(fp)
            n_win = max(1, int(np.ceil(len(wav) / WINDOW_SAMPLES)))
            specs = []
            ends = []
            for wi in range(n_win):
                start = wi * WINDOW_SAMPLES
                chunk = wav[start:start+WINDOW_SAMPLES]
                if len(chunk) < WINDOW_SAMPLES:
                    pad = np.zeros(WINDOW_SAMPLES, dtype=np.float32)
                    pad[:len(chunk)] = chunk
                    chunk = pad
                spec = compute_log_mel_spectrogram(chunk, SAMPLE_RATE)
                specs.append(spec)
                ends.append((wi+1)*WINDOW_SECONDS)
            # batch and predict
            preds_parts = []
            for i in range(0, len(specs), BATCH):
                batch = np.stack(specs[i:i+BATCH])
                bt = torch.from_numpy(batch).float()
                with torch.no_grad():
                    logits = model(bt)
                    probs = torch.sigmoid(logits).numpy()
                preds_parts.append(probs)
            preds = np.concatenate(preds_parts, axis=0) if preds_parts else np.zeros((len(specs), num_classes), dtype=np.float32)
            # map to submission columns
            out_rows = np.zeros((len(ends), len(submission_cols)), dtype=np.float32)
            for j, col in enumerate(submission_cols):
                if col in mapping:
                    out_rows[:, j] = preds[:, mapping[col]]
                else:
                    # unmapped columns -> leave zeros
                    pass
            for j, e in enumerate(ends):
                rid = f'{fp.stem}_{e}'
                rows.append((rid, out_rows[j]))

        # assemble DataFrame and write
        df_rows = []
        for rid, arr in rows:
            row = [rid] + list(map(float, arr.tolist()))
            df_rows.append(row)
        df = pd.DataFrame(df_rows, columns=['row_id'] + submission_cols)
        OUT.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(str(OUT), index=False)
        print(f'Wrote {OUT} ({len(df)} rows) in {time.time()-t0:.1f}s')

except Exception as _err:
    import traceback
    tb = traceback.format_exc()
    LOG = Path('/kaggle/working/notebook_error.log')
    try:
        LOG.write_text(tb)
    except Exception:
        pass
    print('Notebook failed; traceback written to', str(LOG))
    # Ensure a submission.csv exists no matter what (header-only fallback)
    try:
        OUT.parent.mkdir(parents=True, exist_ok=True)
        submission_cols_local = None
        try:
            submission_cols_local = globals().get('submission_cols', None)
        except Exception:
            submission_cols_local = None
        if submission_cols_local is None:
            try:
                sample = pd.read_csv(str(SAMPLE_SUB)).head(0)
                submission_cols_local = [c for c in sample.columns if c != 'row_id']
            except Exception:
                submission_cols_local = None
        if submission_cols_local is None:
            try:
                submission_cols_local = globals().get('labels', []) or []
            except Exception:
                submission_cols_local = []
        submission_cols_local = [str(c) for c in submission_cols_local]
        df_empty = pd.DataFrame(columns=['row_id'] + submission_cols_local)
        df_empty.to_csv(str(OUT), index=False)
        print(f'Wrote fallback submission {OUT} (0 rows) after failure')
    except Exception as _werr:
        print('Failed to write fallback submission:', _werr)
    raise


Using num_classes from checkpoint fc: 215
num_classes=215, len(labels)=215, submission_cols=234, mapped=215, unmapped=19
Wrote results/submission.csv (240 rows) in 2.5s


In [4]:
print('ck_num_classes:', ck_num_classes)
print('num_classes used:', num_classes)
print('len(labels):', len(labels))

print('fc key:', wkey)
if wkey is not None:
    print('state[fc].shape:', state[wkey].shape)


print('labels[:5]', labels[:5])
print('labels[-5:]', labels[-5:])
print('submission_cols[210:220]:', submission_cols[210:220])
for c in submission_cols[210:220]:
    print(c, 'in labels?', c in labels, 'label_index(if present):', (labels.index(c) if c in labels else None))

print('preds.shape:', preds.shape)           # after you compute preds
bad_cols = [c for c,m in mapping.items() if m >= preds.shape[1]]
print('mapping entries >= preds.shape[1]:', bad_cols)

ck_num_classes: 215
num_classes used: 215
len(labels): 215
fc key: fc.weight
state[fc].shape: torch.Size([215, 128])
labels[:5] ['116570', '1491113', '1595929', '22956', '22961']
labels[-5:] ['yebcar', 'yebela1', 'yecmac', 'yecpar', 'yeofly1']
submission_cols[210:220]: ['tattin1', 'thlwre1', 'toctou1', 'trokin', 'trsowl', 'undtin1', 'varant1', 'watjac1', 'wesfie1', 'wfwduc1']
tattin1 in labels? True label_index(if present): 192
thlwre1 in labels? True label_index(if present): 193
toctou1 in labels? True label_index(if present): 194
trokin in labels? True label_index(if present): 195
trsowl in labels? True label_index(if present): 196
undtin1 in labels? True label_index(if present): 197
varant1 in labels? True label_index(if present): 198
watjac1 in labels? True label_index(if present): 199
wesfie1 in labels? True label_index(if present): 200
wfwduc1 in labels? True label_index(if present): 201
preds.shape: (12, 215)
mapping entries >= preds.shape[1]: []
